In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=False)
context

# FLenQA lens drift as prompts grow

This notebook separates shallow answer-interface changes from persistent semantic drift as prompt length grows. It first reproduces the lens's `final_prompt` result, then uses saved fact, question, padding, and final-prompt positions to ask whether drift survives answer-token removal and persists within matched problems. Token prominence is **reciprocal rank** in the saved top-25; a token outside the top-25 contributes zero. This is a descriptive top-25 analysis, not the model's full probability distribution.

The cells are intentionally small: paths, loading, aggregation, metrics, and plots are separate so intermediate analysis can be inserted easily.

In [ ]:
import math
import string

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm
from transformers import AutoTokenizer

from experiments.jlens_readout_sanity.constants import MODEL_PATH

FULL_RUN = context.runs_dir / "flenqa-full-run"
ACCURACY_PATH = context.runs_dir / "flenqa-accuracy" / "results.parquet"
FULL_RUN, ACCURACY_PATH

In [ ]:
def require_same_prompt_ids(reference, **tables):
    reference = set(reference)
    for name, values in tables.items():
        values = set(values)
        missing = len(reference - values)
        extra = len(values - reference)
        if missing or extra:
            raise ValueError(
                f"{name} has {missing} missing and {extra} extra prompt IDs"
            )


def require_same_position_keys(reference, observed, *, name, unit="labeled positions"):
    reference = set(reference)
    observed = set(observed)
    missing = len(reference - observed)
    extra = len(observed - reference)
    if missing or extra:
        raise ValueError(f"{name} has {missing} missing and {extra} extra {unit}")

In [ ]:
accuracy = pd.read_parquet(
    ACCURACY_PATH,
    columns=["prompt_id", "ctx_size", "n_input_tokens", "correct"],
)
display(accuracy.head())
display(accuracy.groupby("ctx_size")["n_input_tokens"].agg(["min", "median", "max"]))
accuracy.shape

In [ ]:
prompts = pd.read_parquet(
    FULL_RUN / "prompts",
    columns=["prompt_id", "problem_id", "task", "label"],
)
positions = pd.read_parquet(
    FULL_RUN / "positions", columns=["prompt_id", "position", "label"]
).rename(columns={"label": "position_label"})
final_positions = (
    positions[positions["position_label"] == "final_prompt"]
    .drop(columns="position_label")
    .rename(columns={"position": "final_position"})
)

require_same_prompt_ids(
    prompts["prompt_id"],
    accuracy=accuracy["prompt_id"],
    final_positions=final_positions["prompt_id"],
)
prompt_info = prompts.merge(accuracy, on="prompt_id", validate="one_to_one").merge(
    final_positions, on="prompt_id", validate="one_to_one"
)
position_info = positions.merge(
    prompt_info[["prompt_id", "problem_id", "ctx_size"]],
    on="prompt_id",
    validate="many_to_one",
)
display(prompt_info.head())
display(prompt_info.groupby("ctx_size")["prompt_id"].size())
position_info.groupby(["position_label", "ctx_size"]).size().unstack(fill_value=0)

In [ ]:
def summarize_tokens(topk, prompt_info):
    """Summarize final-position tokens and return the prompts observed."""
    rows = topk.merge(
        prompt_info, on="prompt_id", how="left", validate="many_to_one", indicator=True
    )
    if (rows["_merge"] != "both").any():
        raise ValueError("top-k rows reference unknown prompt IDs")
    rows = rows[rows["position"] == rows["final_position"]].copy()
    rows["rank_score"] = 1 / rows["rank"]

    keys = ["lens_kind", "layer", "ctx_size", "token_id"]
    summary = rows.groupby(keys, as_index=False).agg(
        appearances=("token_id", "size"),
        rank_score=("rank_score", "sum"),
        logit_sum=("logit", "sum"),
    )
    return summary, set(rows["prompt_id"])

In [ ]:
def summarize_position_tokens(topk, position_info, *, retain_prompt=False):
    """Summarize every labeled position while retaining problem identity."""
    rows = topk.merge(
        position_info,
        on=["prompt_id", "position"],
        how="left",
        validate="many_to_many",
        indicator=True,
    )
    if (rows["_merge"] != "both").any():
        raise ValueError("top-k rows reference unknown labeled positions")
    rows["rank_score"] = 1 / rows["rank"]

    keys = [
        "lens_kind",
        "layer",
        "position_label",
        "ctx_size",
        "problem_id",
        "token_id",
    ]
    if retain_prompt:
        keys.insert(0, "prompt_id")
    summary = rows.groupby(keys, as_index=False).agg(
        appearances=("token_id", "size"),
        rank_score=("rank_score", "sum"),
        logit_sum=("logit", "sum"),
    )
    return summary, set(rows["prompt_id"])

In [ ]:
def combine_token_summaries(summaries):
    keys = ["lens_kind", "layer", "ctx_size", "token_id"]
    return (
        pd.concat(summaries)
        .groupby(keys, as_index=False)[["appearances", "rank_score", "logit_sum"]]
        .sum()
    )

In [ ]:
def combine_position_summaries(summaries, *, retain_problem=True, retain_prompt=False):
    keys = [
        "lens_kind",
        "layer",
        "position_label",
        "ctx_size",
        "token_id",
    ]
    if retain_problem:
        keys.insert(-1, "problem_id")
    if retain_prompt:
        keys.insert(0, "prompt_id")
    return (
        pd.concat(summaries)
        .groupby(keys, as_index=False)[["appearances", "rank_score", "logit_sum"]]
        .sum()
    )

In [ ]:
def measure_distribution_drift(token_stats):
    """Compare each length's token distribution with the shortest length."""
    baseline = token_stats["ctx_size"].min()
    records = []

    for (lens_kind, layer), rows in token_stats.groupby(["lens_kind", "layer"]):
        scores = rows.pivot_table(
            index="token_id", columns="ctx_size", values="rank_score", fill_value=0
        )
        shares = scores / scores.sum()
        distances = 0.5 * shares.sub(shares[baseline], axis=0).abs().sum()
        records.extend(
            {
                "lens_kind": lens_kind,
                "layer": layer,
                "ctx_size": length,
                "total_variation": distance,
            }
            for length, distance in distances.items()
        )

    return pd.DataFrame(records)

In [ ]:
def measure_grouped_distribution_drift(
    token_stats,
    *,
    group_keys,
    expected_ctx_sizes=None,
    require_complete=True,
):
    """Measure TV from each group's earliest observed prompt length."""
    if expected_ctx_sizes is None:
        expected_ctx_sizes = sorted(token_stats["ctx_size"].unique())
    score_keys = [*group_keys, "ctx_size", "token_id"]
    scores = (
        token_stats.groupby(score_keys, as_index=False)["rank_score"]
        .sum()
        .rename(columns={"rank_score": "score"})
    )
    totals = scores.groupby([*group_keys, "ctx_size"])["score"].transform("sum")
    scores["share"] = scores["score"] / totals

    contexts = scores[[*group_keys, "ctx_size"]].drop_duplicates()
    expected_contexts = (
        contexts[group_keys]
        .drop_duplicates()
        .merge(pd.DataFrame({"ctx_size": expected_ctx_sizes}), how="cross")
    )
    context_coverage = expected_contexts.merge(
        contexts,
        on=[*group_keys, "ctx_size"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
    if require_complete and (context_coverage["_merge"] != "both").any():
        raise ValueError("every drift group must contain every length")
    group_baselines = (
        contexts.groupby(group_keys, as_index=False)["ctx_size"]
        .min()
        .rename(columns={"ctx_size": "baseline_ctx_size"})
    )
    scores = scores.merge(group_baselines, on=group_keys, validate="many_to_one")

    baseline_scores = scores[scores["ctx_size"] == scores["baseline_ctx_size"]][
        [*group_keys, "baseline_ctx_size", "token_id", "share"]
    ].rename(columns={"share": "baseline_share"})
    distances = []
    for ctx_size in sorted(scores["ctx_size"].unique()):
        current = scores[scores["ctx_size"] == ctx_size][
            [*group_keys, "baseline_ctx_size", "token_id", "share"]
        ]
        comparison_keys = [*group_keys, "baseline_ctx_size"]
        present_groups = current[comparison_keys].drop_duplicates()
        reference = present_groups.merge(
            baseline_scores,
            on=comparison_keys,
            how="left",
            validate="many_to_many",
        )
        compared = reference.merge(
            current,
            on=[*comparison_keys, "token_id"],
            how="outer",
            validate="one_to_one",
        )
        compared[["baseline_share", "share"]] = compared[
            ["baseline_share", "share"]
        ].fillna(0)
        compared["absolute_change"] = (
            compared["share"] - compared["baseline_share"]
        ).abs()
        distance = (
            compared.groupby(comparison_keys, as_index=False)["absolute_change"]
            .sum()
            .rename(columns={"absolute_change": "total_variation"})
        )
        distance["ctx_size"] = ctx_size
        distance["total_variation"] *= 0.5
        distances.append(distance)
    return pd.concat(distances, ignore_index=True)

In [ ]:
def measure_prompt_distribution_drift(
    prompt_stats, baseline_stats, *, require_complete=True
):
    """Compare each prompt component with its problem's baseline component."""
    component_keys = ["problem_id", "lens_kind", "layer", "position_label"]
    prompt_keys = ["prompt_id", "ctx_size", *component_keys]
    prompt_scores = (
        prompt_stats.groupby([*prompt_keys, "token_id"], as_index=False)["rank_score"]
        .sum()
        .rename(columns={"rank_score": "score"})
    )
    prompt_totals = prompt_scores.groupby(prompt_keys)["score"].transform("sum")
    prompt_scores["share"] = prompt_scores["score"] / prompt_totals

    baseline_scores = (
        baseline_stats.groupby([*component_keys, "token_id"], as_index=False)[
            "rank_score"
        ]
        .sum()
        .rename(columns={"rank_score": "score"})
    )
    baseline_totals = baseline_scores.groupby(component_keys)["score"].transform("sum")
    baseline_scores["baseline_share"] = baseline_scores["score"] / baseline_totals

    prompt_components = prompt_scores[prompt_keys].drop_duplicates()
    baseline_components = baseline_scores[component_keys].drop_duplicates()
    coverage = prompt_components.merge(
        baseline_components,
        on=component_keys,
        how="left",
        validate="many_to_one",
        indicator=True,
    )
    if require_complete and (coverage["_merge"] != "both").any():
        raise ValueError("prompt components are missing baselines")
    prompt_components = coverage[coverage["_merge"] == "both"][prompt_keys]
    prompt_scores = prompt_scores.merge(
        prompt_components, on=prompt_keys, validate="many_to_one"
    )

    reference = prompt_components.merge(
        baseline_scores[[*component_keys, "token_id", "baseline_share"]],
        on=component_keys,
        how="left",
        validate="many_to_many",
    )
    compared = reference.merge(
        prompt_scores[[*prompt_keys, "token_id", "share"]],
        on=[*prompt_keys, "token_id"],
        how="outer",
        validate="one_to_one",
    )
    compared[["baseline_share", "share"]] = compared[
        ["baseline_share", "share"]
    ].fillna(0)
    compared["absolute_change"] = (compared["share"] - compared["baseline_share"]).abs()
    drift = compared.groupby(prompt_keys, as_index=False)["absolute_change"].sum()
    drift["total_variation"] = 0.5 * drift.pop("absolute_change")
    return drift

In [ ]:
SEMANTIC_POSITION_LABELS = frozenset({"fact_a_end", "fact_b_end", "question_end"})


def summarize_persistent_semantic_drift(
    problem_drift,
    *,
    lower_fraction=0.25,
    upper_fraction=0.75,
    layer_min=None,
    layer_max=None,
):
    """Median matched drift across semantic positions and middle layers."""
    if (layer_min is None) != (layer_max is None):
        raise ValueError("layer_min and layer_max must be provided together")
    if layer_min is None:
        max_layer = int(problem_drift["layer"].max())
        layer_min = math.ceil(max_layer * lower_fraction)
        layer_max = math.floor(max_layer * upper_fraction)
    selected = problem_drift[
        problem_drift["position_label"].isin(SEMANTIC_POSITION_LABELS)
        & problem_drift["layer"].between(layer_min, layer_max)
    ]
    summary = (
        selected.groupby(["lens_kind", "problem_id", "ctx_size"], as_index=False)[
            "total_variation"
        ]
        .agg(persistent_drift="median", component_count="size")
        .assign(layer_min=layer_min, layer_max=layer_max)
    )
    return summary

In [ ]:
def measure_drift_error_association(persistent_drift, problem_accuracy):
    """Associate matched persistent drift with problem-level error rate."""
    rows = persistent_drift.merge(
        problem_accuracy,
        on=["problem_id", "ctx_size"],
        how="inner",
        validate="many_to_one",
    )
    rows["error_rate"] = 1 - rows["accuracy"]
    baseline = problem_accuracy["ctx_size"].min()
    baseline_accuracy = problem_accuracy[problem_accuracy["ctx_size"] == baseline][
        ["problem_id", "accuracy"]
    ].rename(columns={"accuracy": "baseline_accuracy"})
    rows = rows.merge(
        baseline_accuracy, on="problem_id", how="left", validate="many_to_one"
    )
    rows["accuracy_loss"] = rows["baseline_accuracy"] - rows["accuracy"]

    def rank_correlation(group, column):
        if group["persistent_drift"].nunique() < 2 or group[column].nunique() < 2:
            return float("nan")
        return (
            group["persistent_drift"]
            .rank(method="average")
            .corr(group[column].rank(method="average"))
        )

    records = []
    for (lens_kind, ctx_size), group in rows.groupby(["lens_kind", "ctx_size"]):
        records.append(
            {
                "lens_kind": lens_kind,
                "ctx_size": ctx_size,
                "problem_count": group["problem_id"].nunique(),
                "spearman_drift_vs_error": rank_correlation(group, "error_rate"),
                "spearman_drift_vs_accuracy_loss": rank_correlation(
                    group, "accuracy_loss"
                ),
            }
        )
    return pd.DataFrame(records)

## Summarize the large top-k table

The Parquet files are read in bounded batches. Each shard is reduced before the next one is read, so raw top-k rows do not accumulate in memory.

In [ ]:
topk_columns = [
    "prompt_id",
    "lens_kind",
    "layer",
    "position",
    "rank",
    "token_id",
    "logit",
]
topk_files = sorted((FULL_RUN / "topk").glob("*.parquet"))
if not topk_files:
    raise FileNotFoundError(f"No top-k shards found in {FULL_RUN / 'topk'}")

prompt_keys = prompt_info[["prompt_id", "ctx_size", "final_position"]]
position_keys = position_info[
    ["prompt_id", "problem_id", "ctx_size", "position", "position_label"]
]
baseline_length = int(prompt_info["ctx_size"].min())
seen_topk_ids = set()
seen_topk_position_keys = set()
token_stats = None
position_token_stats = None
baseline_semantic_token_stats = None
for path in tqdm(topk_files, desc="Reading top-k shards"):
    batch_summaries = []
    position_batch_summaries = []
    baseline_batch_summaries = []
    batches = pq.ParquetFile(path).iter_batches(
        batch_size=250_000, columns=topk_columns
    )
    for batch in batches:
        topk = batch.to_pandas()
        summary, batch_ids = summarize_tokens(topk, prompt_keys)
        position_summary, _ = summarize_position_tokens(topk, position_keys)
        batch_summaries.append(summary)
        position_batch_summaries.append(
            combine_position_summaries([position_summary], retain_problem=False)
        )
        batch_semantic_baseline = position_summary[
            (position_summary["ctx_size"] == baseline_length)
            & position_summary["position_label"].isin(SEMANTIC_POSITION_LABELS)
        ]
        if not batch_semantic_baseline.empty:
            baseline_batch_summaries.append(batch_semantic_baseline)
        seen_topk_ids.update(batch_ids)
        seen_topk_position_keys.update(
            topk[["prompt_id", "position"]].itertuples(index=False, name=None)
        )

    shard_summary = combine_token_summaries(batch_summaries)
    shard_position_summary = combine_position_summaries(
        position_batch_summaries, retain_problem=False
    )
    token_stats = (
        shard_summary
        if token_stats is None
        else combine_token_summaries([token_stats, shard_summary])
    )
    position_token_stats = (
        shard_position_summary
        if position_token_stats is None
        else combine_position_summaries(
            [position_token_stats, shard_position_summary], retain_problem=False
        )
    )
    if baseline_batch_summaries:
        shard_semantic_baseline = combine_position_summaries(baseline_batch_summaries)
        baseline_semantic_token_stats = (
            shard_semantic_baseline
            if baseline_semantic_token_stats is None
            else combine_position_summaries(
                [baseline_semantic_token_stats, shard_semantic_baseline]
            )
        )

require_same_prompt_ids(prompt_info["prompt_id"], topk=seen_topk_ids)
require_same_position_keys(
    position_info[["prompt_id", "position"]].itertuples(index=False, name=None),
    seen_topk_position_keys,
    name="top-k positions",
)
del (
    batch,
    batch_summaries,
    baseline_batch_summaries,
    batch_semantic_baseline,
    batches,
    position_batch_summaries,
    position_summary,
    shard_position_summary,
    shard_summary,
    summary,
    topk,
)

In [ ]:
prompt_counts = (
    prompt_info.groupby("ctx_size", as_index=False)["prompt_id"]
    .nunique()
    .rename(columns={"prompt_id": "prompt_count"})
)
token_stats = token_stats.merge(prompt_counts, on="ctx_size")
token_stats["prominence"] = token_stats["rank_score"] / token_stats["prompt_count"]
token_stats["visibility"] = token_stats["appearances"] / token_stats["prompt_count"]
token_stats["mean_visible_logit"] = (
    token_stats["logit_sum"] / token_stats["appearances"]
)
position_token_stats = position_token_stats[
    ["lens_kind", "layer", "position_label", "ctx_size", "token_id", "rank_score"]
]
baseline_semantic_token_stats = baseline_semantic_token_stats[
    [
        "lens_kind",
        "layer",
        "position_label",
        "ctx_size",
        "problem_id",
        "token_id",
        "rank_score",
    ]
]
display(token_stats.head())
token_stats.shape

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
all_token_ids = (
    set(token_stats["token_id"])
    | set(position_token_stats["token_id"])
    | set(baseline_semantic_token_stats["token_id"])
)
token_text = {
    token_id: tokenizer.decode([token_id], clean_up_tokenization_spaces=False)
    for token_id in all_token_ids
}
COMMON_STOPWORDS = set(
    "a an and are as at be by for from in is it of on or that the this to was were with".split()
)


def token_group(text):
    value = text.strip().lower()
    if not value:
        return "whitespace"
    if value in COMMON_STOPWORDS:
        return "common stopword"
    if value.isalpha():
        return "word"
    if value.isdigit():
        return "number"
    if all(character in string.punctuation for character in value):
        return "punctuation"
    return "other"


token_stats["token"] = token_stats["token_id"].map(token_text)
token_stats["token_group"] = token_stats["token"].map(token_group)
del all_token_ids

In [ ]:
ANSWER_INTERFACE_TOKENS = frozenset(
    {"true", "false", "answer", "response", "<think>", "</think>"}
)


def is_answer_surface_token(text):
    """Identify answer-interface, whitespace, and formatting-only pieces."""
    value = text.strip().casefold()
    if not value or value in ANSWER_INTERFACE_TOKENS:
        return True
    if all(character in string.punctuation for character in value):
        return True
    return value.strip(string.punctuation) in ANSWER_INTERFACE_TOKENS

In [ ]:
for frame in (position_token_stats, baseline_semantic_token_stats):
    frame["token"] = frame["token_id"].map(token_text)
    frame["answer_surface"] = frame["token"].map(is_answer_surface_token)

surface_keys = ["lens_kind", "position_label", "ctx_size"]
surface_mass = (
    position_token_stats[position_token_stats["answer_surface"]]
    .groupby(surface_keys, as_index=False)["rank_score"]
    .sum()
    .rename(columns={"rank_score": "surface_rank_mass"})
)
total_mass = (
    position_token_stats.groupby(surface_keys, as_index=False)["rank_score"]
    .sum()
    .rename(columns={"rank_score": "total_rank_mass"})
)
surface_coverage = total_mass.merge(
    surface_mass, on=surface_keys, how="left", validate="one_to_one"
).fillna({"surface_rank_mass": 0})
surface_coverage["surface_rank_mass_share"] = (
    surface_coverage["surface_rank_mass"] / surface_coverage["total_rank_mass"]
)
display(surface_coverage.head())

## General top-25 drift

Total variation is 0 when the normalized top-25 reciprocal-rank mass matches the shortest prompts and approaches 1 as it becomes completely different.

In [ ]:
drift = measure_distribution_drift(token_stats)
lens_kinds = sorted(drift["lens_kind"].unique())
drift_scale_max = drift["total_variation"].max()
fig, axes = plt.subplots(1, len(lens_kinds), figsize=(13, 5), sharey=True)

for axis, lens_kind in zip(axes, lens_kinds, strict=True):
    matrix = drift[drift["lens_kind"] == lens_kind].pivot(
        index="layer", columns="ctx_size", values="total_variation"
    )
    image = axis.imshow(
        matrix,
        aspect="auto",
        origin="lower",
        cmap="magma",
        vmin=0,
        vmax=drift_scale_max,
    )
    axis.set_xticks(range(len(matrix.columns)), matrix.columns)
    axis.set_yticks(range(len(matrix.index)), matrix.index)
    axis.set_title(f"{lens_kind.title()} Lens")
    axis.set_xlabel("Nominal prompt length")
    fig.colorbar(image, ax=axis, label="Top-25 rank-mass drift")

axes[0].set_ylabel("Layer")
fig.suptitle("How much top-25 reciprocal-rank mass changes with prompt length")
plt.tight_layout()
plt.show()

## Does drift survive the answer interface?

The ablated view removes case-insensitive `True`, `False`, `Answer`, `Response`, thinking tags, whitespace, and formatting-only token pieces. Drift is computed separately at fact, question, sampled-padding, and final-prompt positions. Because a saved label can legitimately have no positions at a length, each position group uses its earliest observed length as baseline and missing heatmap tiles remain blank. A deeper effect should survive this removal and appear before the answer boundary.

In [ ]:
POSITION_GROUP_KEYS = ["lens_kind", "layer", "position_label"]
EXPECTED_CONTEXT_SIZES = sorted(prompt_info["ctx_size"].unique())
observed_position_contexts = position_token_stats[
    [*POSITION_GROUP_KEYS, "ctx_size"]
].drop_duplicates()
expected_position_contexts = (
    observed_position_contexts[POSITION_GROUP_KEYS]
    .drop_duplicates()
    .merge(pd.DataFrame({"ctx_size": EXPECTED_CONTEXT_SIZES}), how="cross")
)
position_context_coverage = expected_position_contexts.merge(
    observed_position_contexts,
    on=[*POSITION_GROUP_KEYS, "ctx_size"],
    how="left",
    validate="one_to_one",
    indicator=True,
)
position_coverage = position_context_coverage.groupby(
    ["position_label", "ctx_size"], as_index=False
).agg(
    expected_groups=("_merge", "size"),
    covered_groups=("_merge", lambda values: (values == "both").sum()),
)
position_coverage["coverage_fraction"] = (
    position_coverage["covered_groups"] / position_coverage["expected_groups"]
)
display("Saved-position coverage by length", position_coverage)

position_token_stats_ablated = position_token_stats[
    ~position_token_stats["answer_surface"]
]
require_same_position_keys(
    position_token_stats[[*POSITION_GROUP_KEYS, "ctx_size"]].itertuples(
        index=False, name=None
    ),
    position_token_stats_ablated[[*POSITION_GROUP_KEYS, "ctx_size"]].itertuples(
        index=False, name=None
    ),
    name="aggregate surface ablation",
    unit="position-layer-length groups",
)
position_drift_raw = measure_grouped_distribution_drift(
    position_token_stats,
    group_keys=POSITION_GROUP_KEYS,
    expected_ctx_sizes=EXPECTED_CONTEXT_SIZES,
    require_complete=False,
)
position_drift_ablated = measure_grouped_distribution_drift(
    position_token_stats_ablated,
    group_keys=POSITION_GROUP_KEYS,
    expected_ctx_sizes=EXPECTED_CONTEXT_SIZES,
    require_complete=False,
).rename(columns={"total_variation": "ablated_total_variation"})
position_drift_comparison = position_drift_raw.merge(
    position_drift_ablated,
    on=[*POSITION_GROUP_KEYS, "ctx_size", "baseline_ctx_size"],
    validate="one_to_one",
).rename(columns={"total_variation": "raw_total_variation"})

long_length = int(position_drift_comparison["ctx_size"].max())
long_position_summary = (
    position_drift_comparison[position_drift_comparison["ctx_size"] == long_length]
    .groupby(["lens_kind", "position_label"], as_index=False)[
        ["raw_total_variation", "ablated_total_variation"]
    ]
    .mean()
)
display("Mean drift across layers at the longest prompts", long_position_summary)
display(
    "Answer-interface rank-mass share at the longest prompts",
    surface_coverage[surface_coverage["ctx_size"] == long_length],
)

In [ ]:
position_label_order = [
    label
    for label in (
        "fact_a_end",
        "fact_b_end",
        "question_end",
        "sampled_padding",
        "final_prompt",
    )
    if label in set(position_drift_comparison["position_label"])
]
plot_specs = [
    (lens_kind, metric_name, value_column)
    for lens_kind in lens_kinds
    for metric_name, value_column in (
        ("raw", "raw_total_variation"),
        ("ablated", "ablated_total_variation"),
    )
]
position_scale_max = (
    position_drift_comparison[["raw_total_variation", "ablated_total_variation"]]
    .to_numpy()
    .max()
)
fig, axes = plt.subplots(
    len(position_label_order),
    len(plot_specs),
    figsize=(18, 3.2 * len(position_label_order)),
    sharex=True,
    sharey=True,
    squeeze=False,
)

for row_index, position_label in enumerate(position_label_order):
    for column_index, (lens_kind, metric_name, value_column) in enumerate(plot_specs):
        axis = axes[row_index, column_index]
        matrix = position_drift_comparison.query(
            "position_label == @position_label and lens_kind == @lens_kind"
        ).pivot(index="layer", columns="ctx_size", values=value_column)
        matrix = matrix.reindex(columns=EXPECTED_CONTEXT_SIZES)
        image = axis.imshow(
            matrix,
            aspect="auto",
            origin="lower",
            cmap="magma",
            vmin=0,
            vmax=position_scale_max,
        )
        axis.set_xticks(range(len(matrix.columns)), matrix.columns)
        axis.set_yticks(range(len(matrix.index)), matrix.index)
        if row_index == 0:
            axis.set_title(f"{lens_kind.title()} — {metric_name}")
        if row_index == len(position_label_order) - 1:
            axis.set_xlabel("Nominal prompt length")
    axes[row_index, 0].set_ylabel(f"{position_label}\nLayer")

fig.colorbar(image, ax=axes.ravel().tolist(), label="Top-25 reciprocal-rank TV")
fig.suptitle("Raw versus answer-interface-ablated drift at every position")
plt.show()

## Persistent drift within matched problems

Each non-baseline prompt is compared with its problem's unique shortest-prompt distribution. Prompt-level distances are averaged within `problem_id` and length. The persistent score is the median surface-ablated total variation across fact/question positions and the middle half of layers; sampled padding and `final_prompt` are excluded. If ablation removes all saved top-25 mass from a component, its normalized drift is undefined, so the component is reported in the coverage tables and excluded rather than treated as zero. A second bounded Parquet pass emits only prompt-level distances, so problem-token distributions do not accumulate in memory. Associations are descriptive problem-level Spearman correlations, computed separately at each length.

In [ ]:
max_semantic_layer = int(baseline_semantic_token_stats["layer"].max())
persistent_layer_min = math.ceil(max_semantic_layer * 0.25)
persistent_layer_max = math.floor(max_semantic_layer * 0.75)
component_keys = ["problem_id", "lens_kind", "layer", "position_label"]
prompt_component_keys = ["prompt_id", "ctx_size", *component_keys]
baseline_semantic_middle = baseline_semantic_token_stats[
    baseline_semantic_token_stats["layer"].between(
        persistent_layer_min, persistent_layer_max
    )
]
baseline_semantic_ablated = baseline_semantic_middle[
    ~baseline_semantic_middle["answer_surface"]
]
baseline_component_coverage = (
    baseline_semantic_middle[component_keys]
    .drop_duplicates()
    .merge(
        baseline_semantic_ablated[component_keys].drop_duplicates(),
        on=component_keys,
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)
baseline_surface_coverage = baseline_component_coverage.groupby(
    ["lens_kind", "position_label"], as_index=False
).agg(
    expected_components=("_merge", "size"),
    retained_components=("_merge", lambda values: (values == "both").sum()),
)
baseline_surface_coverage["coverage_fraction"] = (
    baseline_surface_coverage["retained_components"]
    / baseline_surface_coverage["expected_components"]
)
display("Baseline semantic coverage after ablation", baseline_surface_coverage)

lens_values = sorted(position_token_stats["lens_kind"].unique())
middle_layers = sorted(baseline_semantic_middle["layer"].unique())
problem_drift_sums = None
prompt_surface_coverage_summaries = []
for path in tqdm(topk_files, desc="Reading matched semantic top-k"):
    prompt_batch_summaries = []
    batches = pq.ParquetFile(path).iter_batches(
        batch_size=250_000, columns=topk_columns
    )
    for batch in batches:
        topk = batch.to_pandas()
        prompt_summary, _ = summarize_position_tokens(
            topk, position_keys, retain_prompt=True
        )
        prompt_summary = prompt_summary[
            (prompt_summary["ctx_size"] != baseline_length)
            & prompt_summary["position_label"].isin(SEMANTIC_POSITION_LABELS)
            & prompt_summary["layer"].between(
                persistent_layer_min, persistent_layer_max
            )
        ]
        if not prompt_summary.empty:
            prompt_batch_summaries.append(prompt_summary)

    if not prompt_batch_summaries:
        continue
    shard_prompt_stats = combine_position_summaries(
        prompt_batch_summaries, retain_prompt=True
    )
    shard_prompt_ids = set(shard_prompt_stats["prompt_id"])
    expected_positions = position_info[
        position_info["prompt_id"].isin(shard_prompt_ids)
        & position_info["position_label"].isin(SEMANTIC_POSITION_LABELS)
    ][["prompt_id", "problem_id", "ctx_size", "position_label"]]
    expected_components = expected_positions.merge(
        pd.DataFrame({"lens_kind": lens_values}), how="cross"
    ).merge(pd.DataFrame({"layer": middle_layers}), how="cross")
    require_same_position_keys(
        expected_components[prompt_component_keys].itertuples(index=False, name=None),
        shard_prompt_stats[prompt_component_keys].itertuples(index=False, name=None),
        name=f"{path.name} semantic top-k coverage",
        unit="prompt components",
    )
    shard_prompt_stats["token"] = shard_prompt_stats["token_id"].map(token_text)
    shard_prompt_stats["answer_surface"] = shard_prompt_stats["token"].map(
        is_answer_surface_token
    )
    shard_prompt_ablated = shard_prompt_stats[~shard_prompt_stats["answer_surface"]]
    shard_surface_coverage = (
        shard_prompt_stats[prompt_component_keys]
        .drop_duplicates()
        .merge(
            shard_prompt_ablated[prompt_component_keys].drop_duplicates(),
            on=prompt_component_keys,
            how="left",
            validate="one_to_one",
            indicator=True,
        )
    )
    prompt_surface_coverage_summaries.append(
        shard_surface_coverage.groupby(
            ["lens_kind", "position_label", "ctx_size"], as_index=False
        ).agg(
            expected_components=("_merge", "size"),
            retained_components=(
                "_merge",
                lambda values: (values == "both").sum(),
            ),
        )
    )
    shard_prompt_drift = measure_prompt_distribution_drift(
        shard_prompt_ablated,
        baseline_semantic_ablated,
        require_complete=False,
    )
    if shard_prompt_drift.empty:
        continue
    shard_drift_sums = shard_prompt_drift.groupby(
        [*component_keys, "ctx_size"], as_index=False
    )["total_variation"].agg(drift_sum="sum", variant_count="size")
    problem_drift_sums = (
        shard_drift_sums
        if problem_drift_sums is None
        else pd.concat([problem_drift_sums, shard_drift_sums])
        .groupby([*component_keys, "ctx_size"], as_index=False)[
            ["drift_sum", "variant_count"]
        ]
        .sum()
    )

prompt_surface_coverage = (
    pd.concat(prompt_surface_coverage_summaries)
    .groupby(["lens_kind", "position_label", "ctx_size"], as_index=False)[
        ["expected_components", "retained_components"]
    ]
    .sum()
)
prompt_surface_coverage["coverage_fraction"] = (
    prompt_surface_coverage["retained_components"]
    / prompt_surface_coverage["expected_components"]
)
display("Prompt semantic coverage after ablation", prompt_surface_coverage)

problem_drift = problem_drift_sums.copy()
problem_drift["total_variation"] = (
    problem_drift["drift_sum"] / problem_drift["variant_count"]
)
problem_drift = problem_drift.drop(columns=["drift_sum", "variant_count"])
baseline_problem_drift = baseline_semantic_ablated[component_keys].drop_duplicates()
baseline_problem_drift["ctx_size"] = baseline_length
baseline_problem_drift["total_variation"] = 0.0
problem_drift = pd.concat([baseline_problem_drift, problem_drift], ignore_index=True)
expected_matched_components = baseline_problem_drift[component_keys].merge(
    pd.DataFrame({"ctx_size": sorted(prompt_info["ctx_size"].unique())}),
    how="cross",
)
matched_component_coverage = expected_matched_components.merge(
    problem_drift[[*component_keys, "ctx_size"]].drop_duplicates(),
    on=[*component_keys, "ctx_size"],
    how="left",
    validate="one_to_one",
    indicator=True,
)
matched_drift_coverage = matched_component_coverage.groupby(
    ["lens_kind", "ctx_size"], as_index=False
).agg(
    expected_components=("_merge", "size"),
    retained_components=("_merge", lambda values: (values == "both").sum()),
)
matched_drift_coverage["coverage_fraction"] = (
    matched_drift_coverage["retained_components"]
    / matched_drift_coverage["expected_components"]
)
display("Matched component coverage used by persistent drift", matched_drift_coverage)
persistent_drift = summarize_persistent_semantic_drift(
    problem_drift,
    layer_min=persistent_layer_min,
    layer_max=persistent_layer_max,
)
problem_accuracy = (
    prompt_info.groupby(["problem_id", "ctx_size"], as_index=False)["correct"]
    .mean()
    .rename(columns={"correct": "accuracy"})
)
drift_error_association = measure_drift_error_association(
    persistent_drift, problem_accuracy
)
persistent_summary = persistent_drift.groupby(
    ["lens_kind", "ctx_size"], as_index=False
).agg(
    problem_count=("problem_id", "nunique"),
    mean_persistent_drift=("persistent_drift", "mean"),
    median_persistent_drift=("persistent_drift", "median"),
    lower_quartile=("persistent_drift", lambda values: values.quantile(0.25)),
    upper_quartile=("persistent_drift", lambda values: values.quantile(0.75)),
)
display(persistent_summary)
display(
    "Within-length association with errors and matched accuracy loss",
    drift_error_association[drift_error_association["ctx_size"] != baseline_length],
)

In [ ]:
accuracy_by_length_for_match = prompt_info.groupby("ctx_size", as_index=False)[
    "correct"
].mean()
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for lens_kind, rows in persistent_summary.groupby("lens_kind"):
    axes[0].plot(
        rows["ctx_size"],
        rows["median_persistent_drift"],
        marker="o",
        label=lens_kind,
    )
    axes[0].fill_between(
        rows["ctx_size"],
        rows["lower_quartile"],
        rows["upper_quartile"],
        alpha=0.15,
    )
axes[0].set(
    title="Matched persistent semantic drift",
    xlabel="Prompt length",
    ylabel="Median surface-ablated TV across problems",
)
axes[0].legend()
axes[1].plot(
    accuracy_by_length_for_match["ctx_size"],
    accuracy_by_length_for_match["correct"],
    marker="o",
)
axes[1].set(
    title="Unique-prompt accuracy",
    xlabel="Prompt length",
    ylabel="Accuracy",
)
for axis in axes:
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

baseline_problem_accuracy = problem_accuracy[
    problem_accuracy["ctx_size"] == baseline_length
][["problem_id", "accuracy"]].rename(columns={"accuracy": "baseline_accuracy"})
linked_drift = persistent_drift.merge(
    problem_accuracy,
    on=["problem_id", "ctx_size"],
    validate="many_to_one",
).merge(baseline_problem_accuracy, on="problem_id", validate="many_to_one")
linked_drift["accuracy_loss"] = (
    linked_drift["baseline_accuracy"] - linked_drift["accuracy"]
)
long_linked = linked_drift[linked_drift["ctx_size"] == long_length]
fig, axes = plt.subplots(1, len(lens_kinds), figsize=(13, 4.5), sharey=True)
for axis, lens_kind in zip(axes, lens_kinds, strict=True):
    rows = long_linked[long_linked["lens_kind"] == lens_kind]
    rho = drift_error_association.query(
        "lens_kind == @lens_kind and ctx_size == @long_length"
    )["spearman_drift_vs_accuracy_loss"].iloc[0]
    axis.scatter(rows["persistent_drift"], rows["accuracy_loss"], alpha=0.35)
    axis.set(
        title=f"{lens_kind.title()} Lens — Spearman {rho:.3f}",
        xlabel="Persistent semantic drift",
        ylabel=f"Accuracy loss vs {baseline_length}",
    )
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Which tokens changed?

Choose a lens and layer here. The default uses the final Jacobian Lens layer; edit these two values to explore another slice.

In [ ]:
LENS_KIND = "jacobian"
LAYER = int(token_stats["layer"].max())

selected = token_stats.query("lens_kind == @LENS_KIND and layer == @LAYER")
SHORT_LENGTH = int(selected["ctx_size"].min())
LONG_LENGTH = int(selected["ctx_size"].max())
LENS_KIND, LAYER, SHORT_LENGTH, LONG_LENGTH

In [ ]:
prominence = selected.pivot_table(
    index="token_id", columns="ctx_size", values="prominence", fill_value=0
)
visibility = selected.pivot_table(
    index="token_id", columns="ctx_size", values="visibility", fill_value=0
)
visible_logit = selected.pivot_table(
    index="token_id", columns="ctx_size", values="mean_visible_logit"
)

changes = pd.DataFrame(index=prominence.index)
changes["token"] = changes.index.map(token_text)
changes["token_group"] = changes["token"].map(token_group)
changes["short_prominence"] = prominence[SHORT_LENGTH]
changes["long_prominence"] = prominence[LONG_LENGTH]
changes["prominence_change"] = prominence[LONG_LENGTH] - prominence[SHORT_LENGTH]
changes["visibility_change"] = visibility[LONG_LENGTH] - visibility[SHORT_LENGTH]
changes["conditional_logit_change"] = (
    visible_logit[LONG_LENGTH] - visible_logit[SHORT_LENGTH]
)
risers = changes.nlargest(15, "prominence_change").reset_index()
fallers = changes.nsmallest(15, "prominence_change").reset_index()

In [ ]:
columns = [
    "token_id",
    "token",
    "token_group",
    "short_prominence",
    "long_prominence",
    "prominence_change",
    "visibility_change",
    "conditional_logit_change",
]
print("Logit change is conditional on top-25 visibility at both endpoint lengths.")
display("Tokens rising with prompt length", risers[columns])
display("Tokens falling with prompt length", fallers[columns])

In [ ]:
interesting_ids = [*risers["token_id"].head(4), *fallers["token_id"].head(4)]
trends = selected[selected["token_id"].isin(interesting_ids)].pivot_table(
    index="ctx_size", columns="token_id", values="prominence", fill_value=0
)

plt.figure(figsize=(10, 5))
for token_id in trends.columns:
    plt.plot(
        trends.index, trends[token_id], marker="o", label=repr(token_text[token_id])
    )
plt.xlabel("Nominal prompt length")
plt.ylabel("Mean reciprocal-rank prominence")
plt.title(f"Largest token changes — {LENS_KIND} lens, layer {LAYER}")
plt.legend(ncol=2)
plt.grid(alpha=0.25)
plt.show()

In [ ]:
group_scores = selected.pivot_table(
    index="ctx_size",
    columns="token_group",
    values="rank_score",
    aggfunc="sum",
    fill_value=0,
)
group_share = group_scores.div(group_scores.sum(axis=1), axis=0)
display(group_share)
group_share.plot(figsize=(10, 5), marker="o")
plt.xlabel("Nominal prompt length")
plt.ylabel("Share of top-k rank mass")
plt.title(f"Token groups — {LENS_KIND} lens, layer {LAYER}")
plt.grid(alpha=0.25)
plt.show()

## Put drift beside model accuracy

Accuracy is context only—it is not part of the drift definition.

In [ ]:
accuracy_by_length = (
    prompt_info.groupby("ctx_size", as_index=False)["correct"]
    .mean()
    .rename(columns={"correct": "accuracy"})
)
mean_drift = drift.groupby(["lens_kind", "ctx_size"], as_index=False)[
    "total_variation"
].mean()
accuracy_context = mean_drift.merge(accuracy_by_length, on="ctx_size")
display(accuracy_context)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(accuracy_by_length["ctx_size"], accuracy_by_length["accuracy"], marker="o")
axes[0].set(title="Unique-prompt accuracy", xlabel="Prompt length", ylabel="Accuracy")
for lens_kind, rows in mean_drift.groupby("lens_kind"):
    axes[1].plot(rows["ctx_size"], rows["total_variation"], marker="o", label=lens_kind)
axes[1].set(
    title="Mean drift across layers",
    xlabel="Prompt length",
    ylabel="Top-25 rank-mass drift",
)
axes[1].legend()
for axis in axes:
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
long_drift = drift.query("lens_kind == @LENS_KIND and ctx_size == @LONG_LENGTH")
most_changed_layer = long_drift.loc[long_drift["total_variation"].idxmax()]
strongest_riser = risers.iloc[0]
strongest_faller = fallers.iloc[0]

print(
    f"Largest {LENS_KIND} top-25 rank-mass drift at length {LONG_LENGTH}: "
    f"layer {int(most_changed_layer.layer)} "
    f"(total variation {most_changed_layer.total_variation:.3f})."
)
print(
    f"At selected layer {LAYER}, strongest riser: {strongest_riser.token!r} "
    f"({strongest_riser.prominence_change:+.4f} prominence)."
)
print(
    f"At selected layer {LAYER}, strongest faller: {strongest_faller.token!r} "
    f"({strongest_faller.prominence_change:+.4f} prominence)."
)
for lens_kind in lens_kinds:
    persistent_row = persistent_summary.query(
        "lens_kind == @lens_kind and ctx_size == @LONG_LENGTH"
    ).iloc[0]
    association_row = drift_error_association.query(
        "lens_kind == @lens_kind and ctx_size == @LONG_LENGTH"
    ).iloc[0]
    print(
        f"{lens_kind.title()} matched persistent semantic drift at "
        f"length {LONG_LENGTH}: median "
        f"{persistent_row.median_persistent_drift:.3f}; Spearman versus "
        f"matched accuracy loss "
        f"{association_row.spearman_drift_vs_accuracy_loss:.3f}."
    )